# 🛒 Zid E-commerce — Store Performance ML Project

**Author:** Mousa Alubaid  
**Dataset:** Real Zid Platform Data — 3,851 Stores  
**Goal:** Predict store daily orders & classify store performance using Machine Learning

---

## 📌 Project Objectives
1. **Exploratory Data Analysis (EDA)** — Understand store performance patterns
2. **Feature Engineering** — Build meaningful features from raw KPI data
3. **Regression Model** — Predict `Average Daily Orders` per store
4. **Classification Model** — Classify stores as High / Medium / Low performers
5. **Shipping Analysis** — Which shipping methods drive better outcomes?
6. **Business Insights** — Actionable recommendations for Zid

## 📦 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, confusion_matrix, accuracy_score
)

# Plot style
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'DejaVu Sans'
COLORS = ['#0077b6', '#00b4d8', '#48cae4', '#90e0ef', '#023e8a']

print('✅ All libraries imported successfully')

## 📂 2. Load Real Zid Data

In [ ]:
# ── Load datasets ──────────────────────────────────────────────
kpi      = pd.read_excel('Store_Performance_KPIs.xlsx', sheet_name=0)
shipping = pd.read_excel('Shipping_Method_Performance.xlsx', sheet_name=0)
cancel   = pd.read_excel('Cancellation___Delay_for_Shipping.xlsx', sheet_name=0)

# ── Clean percentage columns ───────────────────────────────────
pct_cols = ['Order Fulfillment Rate (%)', 'Order Cancellation Rate (%)', 'Customer Retention Rate (%)']
for col in pct_cols:
    kpi[col] = kpi[col].astype(str).str.replace('%', '', regex=False).astype(float)

# ── Rename for convenience ─────────────────────────────────────
kpi.columns = ['store_id', 'total_orders', 'fulfillment_rate',
               'cancellation_rate', 'retention_rate', 'avg_order_value', 'avg_daily_orders']

shipping.columns = ['shipping_method', 'total_orders_processed', 'total_canceled', 'cancellation_rate_pct']

print(f'✅ KPI data:      {kpi.shape[0]:,} stores | {kpi.shape[1]} features')
print(f'✅ Shipping data: {shipping.shape[0]:,} methods')
print(f'✅ Cancel data:   {cancel.shape[0]:,} records')
kpi.head()

## 🔍 3. Exploratory Data Analysis (EDA)

In [ ]:
print('=== KPI Summary Statistics ===')
kpi.describe().round(2)

In [ ]:
print('=== Missing Values ===')
print(kpi.isnull().sum())
print(f'\nDuplicated rows: {kpi.duplicated().sum()}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Zid Store KPIs — EDA Overview', fontsize=16, fontweight='bold', y=1.01)

numeric_cols = ['total_orders', 'fulfillment_rate', 'cancellation_rate',
                'retention_rate', 'avg_order_value', 'avg_daily_orders']
titles = ['Total Orders', 'Fulfillment Rate (%)', 'Cancellation Rate (%)',
          'Retention Rate (%)', 'Avg Order Value ($)', 'Avg Daily Orders']

for ax, col, title in zip(axes.flat, numeric_cols, titles):
    # Remove extreme outliers for visualization only
    data = kpi[col].clip(upper=kpi[col].quantile(0.99))
    ax.hist(data, bins=40, color=COLORS[0], edgecolor='white', alpha=0.85)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(title)
    ax.set_ylabel('Count')
    ax.axvline(kpi[col].median(), color='red', linestyle='--', linewidth=1.5, label=f'Median: {kpi[col].median():.1f}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('eda_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# Correlation matrix
fig, ax = plt.subplots(figsize=(9, 7))
corr = kpi[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='Blues',
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
# Shipping method analysis
top10 = shipping.nlargest(10, 'total_orders_processed')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Shipping Methods — Performance Analysis', fontsize=14, fontweight='bold')

# Volume
axes[0].barh(top10['shipping_method'], top10['total_orders_processed'], color=COLORS[0])
axes[0].set_title('Total Orders Processed (Top 10)')
axes[0].set_xlabel('Orders')
axes[0].invert_yaxis()

# Cancellation rate
colors_bar = ['#e63946' if x > 7 else '#00b4d8' for x in top10['cancellation_rate_pct']]
axes[1].barh(top10['shipping_method'], top10['cancellation_rate_pct'], color=colors_bar)
axes[1].set_title('Cancellation Rate % (Red = High Risk > 7%)')
axes[1].set_xlabel('Cancellation Rate (%)')
axes[1].axvline(7, color='red', linestyle='--', linewidth=1.5, label='7% threshold')
axes[1].legend()
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('shipping_analysis.png', bbox_inches='tight')
plt.show()

print('\n📊 Best shipping method by volume:', top10.iloc[0]['shipping_method'])
print('📊 Lowest cancellation rate:', top10.nsmallest(1, 'cancellation_rate_pct').iloc[0]['shipping_method'])

## ⚙️ 4. Feature Engineering

In [ ]:
df = kpi.copy()

# ── Remove extreme outliers (top 1%) ──────────────────────────
q99 = df['avg_daily_orders'].quantile(0.99)
df  = df[df['avg_daily_orders'] <= q99].copy()
print(f'After outlier removal: {len(df):,} stores (removed {len(kpi)-len(df)} outliers)')

# ── New engineered features ───────────────────────────────────
# Revenue proxy
df['est_daily_revenue'] = df['avg_daily_orders'] * df['avg_order_value']

# Performance score (weighted composite)
df['performance_score'] = (
    df['fulfillment_rate']  * 0.35 +
    df['retention_rate']    * 0.30 +
    (100 - df['cancellation_rate']) * 0.35
)

# Order efficiency
df['order_efficiency'] = df['total_orders'] / (df['avg_daily_orders'] + 1)

# High value store flag
df['is_high_value'] = (df['avg_order_value'] > df['avg_order_value'].median()).astype(int)

print('\n✅ New features created:')
print('  → est_daily_revenue : Estimated daily revenue (orders × order value)')
print('  → performance_score : Weighted KPI composite score')
print('  → order_efficiency  : Total orders relative to daily average')
print('  → is_high_value     : Binary flag for above-median order value')
df[['store_id','performance_score','est_daily_revenue','order_efficiency']].head()

## 🤖 5. Model 1 — Regression: Predict Daily Orders

In [ ]:
# ── Features & Target ─────────────────────────────────────────
feature_cols = ['total_orders', 'fulfillment_rate', 'cancellation_rate',
                'retention_rate', 'avg_order_value', 'performance_score',
                'is_high_value']

X = df[feature_cols]
y = df['avg_daily_orders']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')

# ── Train models ──────────────────────────────────────────────
def eval_regressor(name, model, Xtr, Xte, ytr, yte):
    model.fit(Xtr, ytr)
    preds = model.predict(Xte)
    mae  = mean_absolute_error(yte, preds)
    rmse = np.sqrt(mean_squared_error(yte, preds))
    r2   = r2_score(yte, preds)
    print(f'{name:28s}  MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.4f}')
    return model, preds, dict(name=name, MAE=mae, RMSE=rmse, R2=r2)

print('\n=== Regression Model Comparison ===')
print('-' * 65)
reg_results = []
reg_models  = {}

for name, model, use_scaled in [
    ('Linear Regression',    LinearRegression(),                              True),
    ('Random Forest',        RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1), False),
    ('Gradient Boosting',    GradientBoostingRegressor(n_estimators=100, random_state=42), False),
]:
    Xtr = X_train_sc if use_scaled else X_train
    Xte = X_test_sc  if use_scaled else X_test
    m, preds, res = eval_regressor(name, model, Xtr, Xte, y_train, y_test)
    reg_results.append(res)
    reg_models[name] = (m, preds, use_scaled)

reg_df = pd.DataFrame(reg_results).sort_values('R2', ascending=False)
best_reg = reg_df.iloc[0]['name']
print(f'\n🏆 Best Regression Model: {best_reg}  (R²={reg_df.iloc[0]["R2"]:.4f})')

In [ ]:
# ── Regression Visualizations ─────────────────────────────────
best_model_reg, best_preds_reg, _ = reg_models[best_reg]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'Regression Results — {best_reg}', fontsize=14, fontweight='bold')

# Model comparison bar
axes[0].bar(reg_df['name'], reg_df['R2'], color=COLORS[:3])
axes[0].set_title('R² Score Comparison')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(reg_df['R2']):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')

# Actual vs Predicted
axes[1].scatter(y_test, best_preds_reg, alpha=0.3, color=COLORS[1], s=12)
mn, mx = y_test.min(), y_test.max()
axes[1].plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Perfect prediction')
axes[1].set_title('Actual vs Predicted')
axes[1].set_xlabel('Actual Daily Orders')
axes[1].set_ylabel('Predicted Daily Orders')
axes[1].legend()

# Residuals
residuals = y_test.values - best_preds_reg
axes[2].hist(residuals, bins=40, color=COLORS[2], edgecolor='white')
axes[2].axvline(0, color='red', linestyle='--')
axes[2].set_title('Residual Distribution')
axes[2].set_xlabel('Residual (Actual − Predicted)')

plt.tight_layout()
plt.savefig('regression_results.png', bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance
rf_reg = reg_models['Random Forest'][0]
imp_df = pd.DataFrame({'feature': feature_cols, 'importance': rf_reg.feature_importances_})
imp_df = imp_df.sort_values('importance')

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(imp_df['feature'], imp_df['importance'], color=COLORS[0])
ax.set_title('Feature Importance — Random Forest Regressor', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
for bar, val in zip(bars, imp_df['importance']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('feature_importance_reg.png', bbox_inches='tight')
plt.show()

## 🏷️ 6. Model 2 — Classification: Store Performance Tier

In [ ]:
# ── Create target: Performance Tier ──────────────────────────
q33 = df['avg_daily_orders'].quantile(0.33)
q66 = df['avg_daily_orders'].quantile(0.66)

def tier(x):
    if x <= q33:  return 'Low'
    elif x <= q66: return 'Medium'
    else:          return 'High'

df['performance_tier'] = df['avg_daily_orders'].apply(tier)

print('Performance Tier Distribution:')
print(df['performance_tier'].value_counts())
print(f'\nThresholds:  Low ≤ {q33:.1f} | Medium ≤ {q66:.1f} | High > {q66:.1f} orders/day')

# Visualize
fig, ax = plt.subplots(figsize=(7, 4))
counts = df['performance_tier'].value_counts()
bars = ax.bar(counts.index, counts.values, color=['#e63946','#f4a261','#2a9d8f'])
ax.set_title('Store Performance Tier Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Stores')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{val:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('tier_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Train Classification Models ───────────────────────────────
le = LabelEncoder()
y_cls = le.fit_transform(df['performance_tier'])  # High=0, Low=1, Medium=2
X_cls = df[feature_cols]

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

scaler_c = StandardScaler()
Xc_train_sc = scaler_c.fit_transform(Xc_train)
Xc_test_sc  = scaler_c.transform(Xc_test)

def eval_classifier(name, model, Xtr, Xte, ytr, yte):
    model.fit(Xtr, ytr)
    preds = model.predict(Xte)
    acc = accuracy_score(yte, preds)
    print(f'{name:28s}  Accuracy = {acc:.4f} ({acc*100:.1f}%)')
    return model, preds, acc

print('=== Classification Model Comparison ===')
print('-' * 55)
cls_models = {}

for name, model, use_scaled in [
    ('Logistic Regression',  LogisticRegression(max_iter=500, random_state=42), True),
    ('Decision Tree',        DecisionTreeClassifier(max_depth=8, random_state=42), False),
    ('Random Forest',        RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1), False),
]:
    Xtr = Xc_train_sc if use_scaled else Xc_train
    Xte = Xc_test_sc  if use_scaled else Xc_test
    m, preds, acc = eval_classifier(name, model, Xtr, Xte, yc_train, yc_test)
    cls_models[name] = (m, preds, use_scaled, acc)

In [ ]:
# Best classifier detailed report
best_cls_name = max(cls_models, key=lambda k: cls_models[k][3])
best_cls_model, best_cls_preds, _, best_acc = cls_models[best_cls_name]

print(f'🏆 Best Classifier: {best_cls_name} (Accuracy: {best_acc*100:.1f}%)')
print()
print('=== Classification Report ===')
print(classification_report(yc_test, best_cls_preds, target_names=le.classes_))

# Confusion matrix
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(yc_test, best_cls_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=le.classes_, yticklabels=le.classes_)
ax.set_title(f'Confusion Matrix — {best_cls_name}\nAccuracy: {best_acc*100:.1f}%', fontweight='bold')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', bbox_inches='tight')
plt.show()

## 🚚 7. Shipping Performance Deep Dive

In [ ]:
cancel.columns = ['shipping_name', 'total_orders', 'total_cancellations',
                  'total_delays', 'cancellation_rate', 'delay_rate', 'classification']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Shipping Companies — Risk Analysis', fontsize=14, fontweight='bold')

# Highest cancellations
high_cancel = cancel[cancel['classification'] == 'Highest Cancellations'].nlargest(8, 'cancellation_rate')
axes[0].barh(range(len(high_cancel)), high_cancel['cancellation_rate'], color='#e63946')
axes[0].set_yticks(range(len(high_cancel)))
axes[0].set_yticklabels(high_cancel['shipping_name'], fontsize=9)
axes[0].set_title('Highest Cancellation Rate Companies', color='#e63946')
axes[0].set_xlabel('Cancellation Rate (%)')
axes[0].invert_yaxis()

# Lowest delays (best performers)
low_delay = cancel[cancel['classification'] == 'Lowest Delays'].nsmallest(8, 'delay_rate')
axes[1].barh(range(len(low_delay)), low_delay['delay_rate'], color='#2a9d8f')
axes[1].set_yticks(range(len(low_delay)))
axes[1].set_yticklabels(low_delay['shipping_name'], fontsize=9)
axes[1].set_title('Most Reliable Companies (Lowest Delays)', color='#2a9d8f')
axes[1].set_xlabel('Delay Rate (%)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('shipping_risk_analysis.png', bbox_inches='tight')
plt.show()

## 🎯 8. Predict a New Store

In [ ]:
# Simulate a new store scenario
new_store = pd.DataFrame([{
    'total_orders':      150,
    'fulfillment_rate':  88.0,
    'cancellation_rate': 4.5,
    'retention_rate':    35.0,
    'avg_order_value':   210.0,
    'performance_score': 88*0.35 + 35*0.30 + (100-4.5)*0.35,
    'is_high_value':     1,
}])

# Regression prediction
rf_reg_model = reg_models['Random Forest'][0]
pred_orders  = rf_reg_model.predict(new_store)[0]

# Classification prediction
rf_cls_model = cls_models['Random Forest'][0]
pred_tier_enc = rf_cls_model.predict(new_store)[0]
pred_tier    = le.inverse_transform([pred_tier_enc])[0]

print('=' * 50)
print('   🏪 NEW STORE PREDICTION RESULTS')
print('=' * 50)
print(f'  📦 Predicted Daily Orders : {pred_orders:.1f} orders/day')
print(f'  🏷️  Performance Tier       : {pred_tier}')
print(f'  📅 Estimated Monthly Orders: {pred_orders * 30:.0f}')
print(f'  💰 Estimated Monthly Revenue: SAR {pred_orders * 30 * 210:,.0f}')
print('=' * 50)

## 📝 9. Final Summary & Business Insights

In [ ]:
best_r2  = reg_df.iloc[0]['R2']
best_mae = reg_df.iloc[0]['MAE']

print('=' * 60)
print('   📊 PROJECT SUMMARY — ZID ML ANALYSIS')
print('=' * 60)
print(f'  Dataset             : {len(df):,} Zid stores (real data)')
print(f'  Features Used       : {len(feature_cols)} KPI features')
print()
print('  🔵 REGRESSION MODEL (Predict Daily Orders)')
print(f'     Best Model : {best_reg}')
print(f'     R² Score   : {best_r2:.4f} ({best_r2*100:.1f}% variance explained)')
print(f'     MAE        : ±{best_mae:.2f} orders/day')
print()
print('  🟢 CLASSIFICATION MODEL (Predict Tier)')
print(f'     Best Model : {best_cls_name}')
print(f'     Accuracy   : {best_acc*100:.1f}%')
print()
print('  💡 KEY BUSINESS INSIGHTS')
print('  ─────────────────────────────────────────────────────')

# Dynamic insights from data
top_feature = imp_df.iloc[-1]['feature']
high_stores = df[df['performance_tier'] == 'High']
low_stores  = df[df['performance_tier'] == 'Low']

print(f'  → Most predictive feature: {top_feature}')
print(f'  → High-tier stores avg retention: {high_stores["retention_rate"].mean():.1f}%')
print(f'  → Low-tier stores avg retention:  {low_stores["retention_rate"].mean():.1f}%')
print(f'  → High-tier stores avg cancellation: {high_stores["cancellation_rate"].mean():.1f}%')
print(f'  → Low-tier stores avg cancellation:  {low_stores["cancellation_rate"].mean():.1f}%')
print(f'  → Aramex dominates with {shipping.iloc[0]["total_orders_processed"]:,} orders processed')
print()
print('  📌 RECOMMENDATIONS FOR ZID')
print('  → Focus on improving customer retention to boost daily orders')
print('  → Reduce cancellation rates by streamlining payment & shipping')
print('  → Use this model to flag at-risk stores before they churn')
print('  → Prioritize Aramex & low-cancel carriers for store onboarding')
print('=' * 60)